# Run the Cox Models

In [1]:
import datetime

if not hasattr(datetime, "UTC"):
    datetime.UTC = datetime.timezone.utc

In [ ]:
! pip install lifelines

In [3]:
# Imports here.
import numpy as np
import pandas as pd
import os
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import fdrcorrection
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError

import warnings
warnings.filterwarnings("ignore")

# Get Codes

In [ ]:
codes = pd.read_csv('../../data/labels.csv')
codes = codes[['FinnGen_Phenocode','ICD10_Codes','Cohort','Type','UKB_Description_For_Plots']]
codes = codes[codes['Cohort']=='UKB']
codes

In [ ]:
t = codes['FinnGen_Phenocode'].str.split(',').explode().str.strip().tolist()
t[:10]

In [ ]:
ndd_list = ['AD', 'DEM', 'PD', 'VAS']
# ndd_list = ['AD']
condition_list = t.copy()
print(condition_list)
print(len(condition_list))
print(len(ndd_list))

# Cox: both-sexes model

In [ ]:
# both_sexes
year = '2024'
date = 'JULY_1_2026'
model = 'both-sexes'
lag_list = ['0']

failed_codes = []
results = []
cov_list = []

for ndd in ndd_list:
    
    for lag in lag_list:

            #Load df
            df = pd.read_csv(f'data/{ndd}_JULY_23_2026_ready_cox.csv', parse_dates = True, low_memory = False)
            
            # Find codes to use so we don't have to use EVERYTHING
            codes_with_data = []

            for code in condition_list:

                try:
                    m = df[['age_at_tenure', 'SEX', 'tenure', ndd, f'QC{lag}_{code}', 'APOE']]
    
                    n=sum(m[f'QC{lag}_{code}'])
                    df_pair = m[m[f'QC{lag}_{code}']==1]
                    n_pairs = sum(df_pair[ndd])
                    if n == 0:
                        pass
                    elif n_pairs < 5:
                        pass
                    elif n == n_pairs:
                        pass
                    else:
                        print(code)
                        codes_with_data.append(code)
                        
                except Exception as e:
                    print(f'CODE {code} does not exist')
                    continue

            print(ndd)
            print(len(codes_with_data))

            for code in codes_with_data:  

                try:
                    #m = df[df[f'{code}_exclude']==False]
                    m = df[['age_at_tenure', 'SEX', 'tenure', ndd, f'QC{lag}_{code}', 'APOE']]
                    
                    n=sum(m[f'QC{lag}_{code}'])
                    df_pair = m[m[f'QC{lag}_{code}']==1]
                    n_pairs = sum(df_pair[ndd])

                    formula=f"C(SEX) + C(QC{lag}_{code}) + C(APOE) + age_at_tenure"
                    cph = CoxPHFitter()
                    cph.fit(m, duration_col = 'tenure', event_col = ndd, formula = formula, fit_options = {'step_size':0.1}, show_progress=False)
                    #cph.print_summary()
                    #cph.plot()

                except ConvergenceError:
                    print(f"⚠️  Skipping {code}: model failed to converge.")
                    failed_codes.append(ndd)
                    failed_codes.append(code)
                    failed_codes.append(lag)
                    continue

                except Exception as e:
                    print(f"⚠️  Skipping {code}: unexpected error -> {e}")
                    failed_codes.append(ndd)
                    failed_codes.append(code)
                    failed_codes.append(lag)
                    continue
                    
                # Extract betas and SEs for a variable of interest
                beta = cph.params_[f'C(QC{lag}_{code})[T.1]']
                se = cph.standard_errors_[f'C(QC{lag}_{code})[T.1]']
    
                # extract HR, CI, p for each model
                covariate = code
                summary = cph.summary.loc[f'C(QC{lag}_{code})[T.1]']
                #print(summary)
                HR = summary['exp(coef)']
                ci_min = summary['exp(coef) lower 95%']
                ci_max = summary['exp(coef) upper 95%']
                p = summary['p'] 
    
                print(covariate, ndd, HR, beta, se, ci_min, ci_max, p, n_pairs, n)
                results.append((covariate, ndd, model, lag, HR, beta, se, ci_min, ci_max, p, n_pairs, n))
                
cox1 = pd.DataFrame(results, columns=('PRIOR','OUTCOME', 'MODEL', 'LAG', 'HR', 'beta', 'se', 'ci_min', "ci_max", 'P_VAL', "N_pairs", "N"))

# Cox: female-only

In [ ]:
# female-only

year = '2024'
date = 'JULY_1_2026'
model = 'female-only'
lag_list = ['0']

failed_codes = []
results = []
cov_list = []

for ndd in ndd_list:
    
    for lag in lag_list:

            #Load df
            df = pd.read_csv(f'data/{ndd}_JULY_23_2026_ready_cox.csv', parse_dates = True, low_memory = False)
            df = df[df['SEX']==0]
            
            # Only usable codes
            codes_with_data = []

            for code in condition_list:

                try:
                    m = df[['age_at_tenure', 'tenure', ndd, f'QC{lag}_{code}', 'APOE']]
    
                    n=sum(m[f'QC{lag}_{code}'])
                    df_pair = m[m[f'QC{lag}_{code}']==1]
                    n_pairs = sum(df_pair[ndd])
                    if n == 0:
                        pass
                    elif n_pairs < 5:
                        pass
                    elif n == n_pairs:
                        pass
                    else:
                        print(code)
                        codes_with_data.append(code)
                        
                except Exception as e:
                    print(f'CODE {code} does not exist')
                    continue

            print(ndd)
            print(len(codes_with_data))

            for code in codes_with_data:  

                try:
                    #m = df[df[f'{code}_exclude']==False]
                    m = df[['age_at_tenure', 'tenure', ndd, f'QC{lag}_{code}', 'APOE']]
                    
                    n=sum(m[f'QC{lag}_{code}'])
                    df_pair = m[m[f'QC{lag}_{code}']==1]
                    n_pairs = sum(df_pair[ndd])

                    formula=f"C(QC{lag}_{code}) + C(APOE) + age_at_tenure"
                    cph = CoxPHFitter()
                    cph.fit(m, duration_col = 'tenure', event_col = ndd, formula = formula, fit_options = {'step_size':0.1}, show_progress=False)
                    #cph.print_summary()
                    #cph.plot()

                except ConvergenceError:
                    print(f"⚠️  Skipping {code}: model failed to converge.")
                    failed_codes.append(ndd)
                    failed_codes.append(code)
                    failed_codes.append(lag)
                    continue

                except Exception as e:
                    print(f"⚠️  Skipping {code}: unexpected error -> {e}")
                    failed_codes.append(ndd)
                    failed_codes.append(code)
                    failed_codes.append(lag)
                    continue
                    
                # Extract betas and SEs for a variable of interest
                beta = cph.params_[f'C(QC{lag}_{code})[T.1]']
                se = cph.standard_errors_[f'C(QC{lag}_{code})[T.1]']
    
                # extract HR, CI, p for each model
                covariate = code
                summary = cph.summary.loc[f'C(QC{lag}_{code})[T.1]']
                #print(summary)
                HR = summary['exp(coef)']
                ci_min = summary['exp(coef) lower 95%']
                ci_max = summary['exp(coef) upper 95%']
                p = summary['p'] 
    
                print(covariate, ndd, HR, beta, se, ci_min, ci_max, p, n_pairs, n)
                results.append((covariate, ndd, model, lag, HR, beta, se, ci_min, ci_max, p, n_pairs, n))
                
cox2 = pd.DataFrame(results, columns=('PRIOR','OUTCOME', 'MODEL', 'LAG', 'HR', 'beta', 'se', 'ci_min', "ci_max", 'P_VAL', "N_pairs", "N"))

In [ ]:
cox2

# Cox: male-only

In [ ]:
# male-only

year = '2024'
date = 'JULY_1_2026'
model = 'male-only'
lag_list = ['0']

failed_codes = []
results = []
cov_list = []

for ndd in ndd_list:
    
    for lag in lag_list:

            #Load df
            df = pd.read_csv(f'data/{ndd}_JULY_23_2026_ready_cox.csv', parse_dates = True, low_memory = False)
            df = df[df['SEX']==1]
            
            # Only usable codes
            codes_with_data = []

            for code in condition_list:

                try:
                    m = df[['age_at_tenure', 'tenure', ndd, f'QC{lag}_{code}', 'APOE']]
    
                    n=sum(m[f'QC{lag}_{code}'])
                    df_pair = m[m[f'QC{lag}_{code}']==1]
                    n_pairs = sum(df_pair[ndd])
                    if n == 0:
                        pass
                    elif n_pairs < 5:
                        pass
                    elif n == n_pairs:
                        pass
                    else:
                        print(code)
                        codes_with_data.append(code)
                        
                except Exception as e:
                    print(f'CODE {code} does not exist')
                    continue

            print(ndd)
            print(len(codes_with_data))

            for code in codes_with_data:  

                try:
                    #m = df[df[f'{code}_exclude']==False]
                    m = df[['age_at_tenure', 'tenure', ndd, f'QC{lag}_{code}', 'APOE']]
                    
                    n=sum(m[f'QC{lag}_{code}'])
                    df_pair = m[m[f'QC{lag}_{code}']==1]
                    n_pairs = sum(df_pair[ndd])

                    formula=f"C(QC{lag}_{code}) + C(APOE) + age_at_tenure"
                    cph = CoxPHFitter()
                    cph.fit(m, duration_col = 'tenure', event_col = ndd, formula = formula, fit_options = {'step_size':0.1}, show_progress=False)
                    #cph.print_summary()
                    #cph.plot()

                except ConvergenceError:
                    print(f"⚠️  Skipping {code}: model failed to converge.")
                    failed_codes.append(ndd)
                    failed_codes.append(code)
                    failed_codes.append(lag)
                    continue

                except Exception as e:
                    print(f"⚠️  Skipping {code}: unexpected error -> {e}")
                    failed_codes.append(ndd)
                    failed_codes.append(code)
                    failed_codes.append(lag)
                    continue
                    
                # Extract betas and SEs for a variable of interest
                beta = cph.params_[f'C(QC{lag}_{code})[T.1]']
                se = cph.standard_errors_[f'C(QC{lag}_{code})[T.1]']
    
                # extract HR, CI, p for each model
                covariate = code
                summary = cph.summary.loc[f'C(QC{lag}_{code})[T.1]']
                #print(summary)
                HR = summary['exp(coef)']
                ci_min = summary['exp(coef) lower 95%']
                ci_max = summary['exp(coef) upper 95%']
                p = summary['p'] 
    
                print(covariate, ndd, HR, beta, se, ci_min, ci_max, p, n_pairs, n)
                results.append((covariate, ndd, model, lag, HR, beta, se, ci_min, ci_max, p, n_pairs, n))
                
cox3 = pd.DataFrame(results, columns=('PRIOR','OUTCOME', 'MODEL', 'LAG', 'HR', 'beta', 'se', 'ci_min', "ci_max", 'P_VAL', "N_pairs", "N"))

In [ ]:
cox3

# Cox: Concat and Save Results

In [ ]:
#Combine results
output = pd.concat([cox1,cox2,cox3])

#Adding FDR Correction

#Sort P-values
output = output.sort_values(by = "P_VAL")

#Drop Nan-values
output = output.dropna()

#FDR Correction
rejected, p_corr = fdrcorrection(output['P_VAL'], is_sorted=True)
output['P_CORR'] = p_corr
output['SIGNIFICANT'] = rejected

output

In [13]:
output.to_csv('data/All_NDD_cox_results.csv',index=False)

In [ ]:
output

In [ ]:
t = output[output['PRIOR']=='J10_INFLUPNEU']
t

# Cox: Interaction Term

In [ ]:
# interaction term

year = '2024'
date = 'JULY_23_2026'
model = 'interaction-term'
lag_list = ['0']

failed_codes = []
results = []
cov_list = []

for ndd in ndd_list:
    
    for lag in lag_list:

            #Load df
            df = pd.read_csv(f'data/{ndd}_{date}_ready_cox.csv', parse_dates = True, low_memory = False)
            
            # only usable codes
            codes_with_data = []

            for code in condition_list:

                try:
                    df[f'interactor_{code}_sex'] = (df[f'SEX']) * (df[f'QC0_{code}'])
                    m = df[['age_at_tenure', 'SEX', 'tenure', ndd, f'QC{lag}_' + code, f'interactor_{code}_sex', 'APOE']]
    
                    n=sum(m[f'QC{lag}_{code}'])
                    df_pair = m[m[f'QC{lag}_{code}']==1]
                    n_pairs = sum(df_pair[ndd])
                    if n == 0:
                        pass
                    elif n_pairs < 5:
                        pass
                    elif n == n_pairs:
                        pass
                    else:
                        print(code)
                        codes_with_data.append(code)
                        
                except Exception as e:
                    print(f'CODE {code} does not exist')
                    continue

            print(ndd)
            print(len(codes_with_data))

            for code in codes_with_data:  

                try:
                    df[f'interactor_{code}_sex'] = (df[f'SEX']) * (df[f'QC0_{code}'])
                    m = df[['age_at_tenure', 'SEX', 'tenure', ndd, f'QC{lag}_' + code, f'interactor_{code}_sex', 'APOE']]
                    
                    n=sum(m[f'QC{lag}_{code}'])
                    df_pair = m[m[f'QC{lag}_{code}']==1]
                    n_pairs = sum(df_pair[ndd])

                    formula=f"C(QC{lag}_{code}) + C(SEX) + C(interactor_{code}_sex) + age_at_tenure + C(APOE)"
                    cph = CoxPHFitter()
                    cph.fit(m, duration_col = 'tenure', event_col = ndd, formula = formula, fit_options = {'step_size':0.1}, show_progress=False)
                    # cph.print_summary()
                    #cph.plot()

                except ConvergenceError:
                    print(f"⚠️  Skipping {code}: model failed to converge.")
                    failed_codes.append(ndd)
                    failed_codes.append(code)
                    failed_codes.append(lag)
                    continue

                except Exception as e:
                    print(f"⚠️  Skipping {code}: unexpected error -> {e}")
                    failed_codes.append(ndd)
                    failed_codes.append(code)
                    failed_codes.append(lag)
                    continue
            
                model = f'{code}_sex_interaction'

                #Add sex
                # covariate = f'SEX'
                covariate = f'C(SEX)[T.1]'
                
                # Extract betas and SEs for a variable of interest
                beta = cph.params_[covariate]
                se = cph.standard_errors_[covariate]
    
                # extract HR, CI, p for each model
                summary = cph.summary.loc[covariate]
                HR = summary['exp(coef)']
                ci_min = summary['exp(coef) lower 95%']
                ci_max = summary['exp(coef) upper 95%']
                p = summary['p'] 
                
                print(covariate, ndd, HR, beta, se, ci_min, ci_max, p, n_pairs, n)
                results.append((covariate, ndd, model, lag, HR, beta, se, ci_min, ci_max, p, n_pairs, n))
                
                #Add code
                covariate = code
                
                # Extract betas and SEs for a variable of interest
                beta = cph.params_[f'C(QC{lag}_{code})[T.1]']
                se = cph.standard_errors_[f'C(QC{lag}_{code})[T.1]']
                
                # extract HR, CI, p for each model
                summary = cph.summary.loc[f'C(QC{lag}_{code})[T.1]']
                HR = summary['exp(coef)']
                ci_min = summary['exp(coef) lower 95%']
                ci_max = summary['exp(coef) upper 95%']
                p = summary['p'] 
                
                print(covariate, ndd, HR, beta, se, ci_min, ci_max, p, n_pairs, n)
                results.append((covariate, ndd, model, lag, HR, beta, se, ci_min, ci_max, p, n_pairs, n))
                
                #Add interactor
                # covariate = f'interactor_{code}_sex'
                covariate = f'C(interactor_{code}_sex)[T.1]'
                
                # Extract betas and SEs for a variable of interest
                beta = cph.params_[covariate]
                se = cph.standard_errors_[covariate]
    
                # extract HR, CI, p for each model
                summary = cph.summary.loc[covariate]
                HR = summary['exp(coef)']
                ci_min = summary['exp(coef) lower 95%']
                ci_max = summary['exp(coef) upper 95%']
                p = summary['p'] 
                
                print(covariate, ndd, HR, beta, se, ci_min, ci_max, p, n_pairs, n)
                results.append((covariate, ndd, model, lag, HR, beta, se, ci_min, ci_max, p, n_pairs, n))
                
cox_interaction = pd.DataFrame(results, columns=('PRIOR','OUTCOME', 'MODEL', 'LAG', 'HR', 'beta', 'se', 'ci_min', "ci_max", 'P_VAL', "N_pairs", "N"))

In [ ]:
cox_interaction

In [ ]:
#Adding FDR Correction

#Sort P-values
cox_interaction = cox_interaction.sort_values(by = "P_VAL")

#Drop Nan-values
cox_interaction = cox_interaction.dropna()

#FDR Correction
rejected, p_corr = fdrcorrection(cox_interaction['P_VAL'], is_sorted=True)
cox_interaction['P_CORR'] = p_corr
cox_interaction['SIGNIFICANT'] = rejected

cox_interaction

In [26]:
cox_interaction.to_csv('cox_interaction.csv',index=False)

# Z-score

**note: must run Cox before**

In [20]:
cox = output.copy()

In [ ]:
# modified on Jan 15

results = []
for ndd in ndd_list:
    for code in condition_list:
        # Extract betas and SEs for a variable of interest
        df = cox.copy()
        df = cox[cox['PRIOR'] == code]
        df = df[df['OUTCOME'] == ndd]
        if len(df) < 2:
            continue
        else:
            
            # establishing the males and demale models
            #males
            m = df[df['MODEL'] == 'male-only']
            m = m.reset_index()
            #print(m)

            #females
            f = df[df['MODEL'] == 'female-only']
            f = f.reset_index()
            #print(f)
            
            
            #  if males or females are missing, then skip the case
            if len(m) == 0 or len(f) == 0:
                print(f'skipping {code}: missing male or female data')
                continue
                
            # females
            beta1 = m.loc[0, 'beta']
            #print(beta2)
            se1 = m.loc[0, 'se']
            #print(se2)
            
            beta1 = m.loc[0, 'beta']
            #print(beta2)
            se1 = m.loc[0, 'se']
            #print(se2)
            
            #females
            beta2 = f.loc[0, 'beta']
            #print(beta1)
            se2 = f.loc[0, 'se']
            #print(se1)

            # Compute z-test
            z = (beta1 - beta2) / np.sqrt(se1**2 + se2**2)

            from scipy.stats import norm
            p = 2 * (1 - norm.cdf(abs(z)))  # two-tailed p-value

            print(code, f"Z = {z:.3f}, p = {p:.3g}")

            sig = p<0.05
            
            results.append((code, ndd, z, p, sig))

final = pd.DataFrame(results, columns=('PRIOR','OUTCOME', 'Z', 'P', 'SIG'))  
final    

In [24]:
final.to_csv(f'All_NDD_Z_test.csv',index=False)

# Interaction term GLM model

In [ ]:
ndd_list = ['AD', 'DEM', 'PD']
condition_list
print(condition_list)
print(len(condition_list))
print(len(ndd_list))

In [44]:
date='JULY_23_2026'

In [ ]:
final = pd.DataFrame()

for ndd in ndd_list:
    print(ndd)
    
    for code in condition_list:
        try:
            df = pd.read_csv(f'data/{ndd}_{date}_ready_cox.csv', parse_dates = True, low_memory = False)
            df[f'interactor_{code}_sex'] = (df[f'SEX']) * (df[f'QC0_{code}'])
            variable1 = f'QC0_{code}'
            variable2 = f'SEX'
            interaction_term = f'interactor_{code}_sex'
            model = f'{code} and Genetic_Sex interaction'
            data = df
            this_formula = ndd + f"~ {variable1} + {interaction_term} + {variable2} + age_at_tenure + APOE"
            fitted = sm.formula.glm(formula=this_formula, family=sm.families.Binomial(), data=data).fit()
            #print(fitted.summary())

            list_terms = [f'{variable1}', f'{interaction_term}', f'{variable2}']
            results = []
            for i in list_terms:
                beta_coef  = fitted.params.loc[i]
                beta_se  = fitted.bse.loc[i]
                p_val = fitted.pvalues.loc[i]
                z_val = beta_coef/beta_se
                odds_ratio = np.exp(fitted.params.loc[i])
                conf = fitted.conf_int().loc[i]
                    #m5, m95 = np.exp(conf)
                m5, m95 = conf
                print(i, odds_ratio, beta_coef, beta_se, m5, m95, z_val, p_val)
                results.append((ndd, model, i, odds_ratio, beta_coef, beta_se, m5, m95, z_val, p_val))
            output1 = pd.DataFrame(results, columns=('NDD', 'Model', 'Parameter','OR', 'Beta','SE', '95% CI low', "95% CI high", 'z', "P-value"))
            final = pd.concat([final, output1])
        except Exception as e:
            print(f'COULD NOT FIND {code} RESULT')
            continue
final2 = final.sort_values(by = 'P-value')
final2

In [48]:
final2.to_csv('glm_interaction_term.csv',index=False)